## Loading Environment Variables from the secrets file

In [3]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_KEY")

if API_KEY:
    print("API Key Attained")
else:
    print("API Key Missing")

API Key Attained


## Testing if the API Key Enables Data Access

In [4]:
CITY = "london"

url = f"https://api.waqi.info/feed/{CITY}/?token={API_KEY}"

response = requests.get(url)

data = response.json()

if data['status'] == 'ok':
    print(f"Data Fetched Successfully for {CITY.capitalize()}!")
    
    current_aqi = data['data']['aqi']
    print(f"Current AQI: {current_aqi}")
    
    print("\nAvailable data sections:")
    for key in data['data'].keys():
        print(f"- {key}")
else:
    print(f" Unable to fetch data. Error: {data.get('data', 'Unknown error')}")

Data Fetched Successfully for London!
Current AQI: 32

Available data sections:
- aqi
- idx
- attributions
- city
- dominentpol
- iaqi
- time
- forecast
- debug


## Pulling Relevant Historical Data

In [5]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_historical_aqi_data(city_name: str, years_back: int = 3) -> pd.DataFrame:
    """
    Resolves coordinates for a target city and fetches historical air quality data.
    """
    
    print(f"Resolving coordinates for target city: {city_name.capitalize()}")
    geocode_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}&count=1&format=json"
    
    geo_response = requests.get(geocode_url)
    geo_response.raise_for_status()
    geo_data = geo_response.json()

    if not geo_data.get("results"):
        raise ValueError(f"Coordinates for '{city_name}' could not be resolved.")

    lat = geo_data["results"][0]["latitude"]
    lon = geo_data["results"][0]["longitude"]
    print(f"Coordinates resolved: Latitude {lat}, Longitude {lon}")

    end_date = datetime.now()
    start_date = end_date - timedelta(days=365 * years_back)
    
    print("Retrieving historical dataset...")
    history_url = (
        f"https://air-quality-api.open-meteo.com/v1/air-quality?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start_date.strftime('%Y-%m-%d')}"
        f"&end_date={end_date.strftime('%Y-%m-%d')}"
        f"&hourly=pm10,pm2_5,nitrogen_dioxide,ozone"
    )
    
    hist_response = requests.get(history_url)
    hist_response.raise_for_status()
    hist_data = hist_response.json()
    
    hourly_data = hist_data.get("hourly", {})
    df = pd.DataFrame({
        "timestamp": pd.to_datetime(hourly_data.get("time", [])),
        "pm10": hourly_data.get("pm10", []),
        "pm2_5": hourly_data.get("pm2_5", []),
        "no2": hourly_data.get("nitrogen_dioxide", []),
        "ozone": hourly_data.get("ozone", [])
    })
    
    df = df.dropna().reset_index(drop=True)
    return df

TARGET_CITY = "london"
historical_df = fetch_historical_aqi_data(TARGET_CITY, years_back=3)

print("\nDataset extraction complete.")
print(f"Total historical records extracted: {len(historical_df)}")
print("\nData Preview:")
print(historical_df.head())

Resolving coordinates for target city: London
Coordinates resolved: Latitude 51.50853, Longitude -0.12574
Retrieving historical dataset...

Dataset extraction complete.
Total historical records extracted: 26304

Data Preview:
            timestamp  pm10  pm2_5   no2  ozone
0 2023-04-19 00:00:00  28.2   19.4  11.3   72.0
1 2023-04-19 01:00:00  28.8   19.6   9.9   67.0
2 2023-04-19 02:00:00  27.9   20.6  10.1   63.0
3 2023-04-19 03:00:00  28.3   20.8  11.2   55.0
4 2023-04-19 04:00:00  29.2   22.1  12.1   50.0


## Hopswork Setup

In [ ]:
import time

# Define the global cohort
GLOBAL_CITIES = [
    "london",      
    "beijing",     
    "los angeles", 
    "mumbai",      
    "sydney"       
]

master_data_frames = []

print("Initiating global dataset aggregation pipeline...\n")

for city in GLOBAL_CITIES:
    try:
        # Fetch data using the function from the previous cell
        city_df = fetch_historical_aqi_data(city, years_back=3)
        city_df['city'] = city
        master_data_frames.append(city_df)
        time.sleep(1) # Pause to respect API rate limits
    except Exception as e:
        print(f"Error processing {city.capitalize()}: {str(e)}")

# Concatenate all into one master dataset
master_dataset = pd.concat(master_data_frames, ignore_index=True)

print("\nGlobal dataset construction complete.")
print("-" * 40)
print(f"Total aggregated records: {len(master_dataset)}")
print(f"Cities included: {master_dataset['city'].unique()}")

In [ ]:
import numpy as np

def build_feature_pipeline(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforms raw hourly meteorological data into a daily feature set.
    """
    print("Initiating feature engineering pipeline...")
    df = raw_df.copy()
    
    # Temporal Aggregation: Convert hourly to daily averages
    df['date'] = df['timestamp'].dt.date
    
    daily_df = df.groupby(['city', 'date']).agg({
        'pm10': 'mean',
        'pm2_5': 'mean',
        'no2': 'mean',
        'ozone': 'mean'
    }).reset_index()
    
    daily_df['date'] = pd.to_datetime(daily_df['date'])
    daily_df = daily_df.sort_values(by=['city', 'date']).reset_index(drop=True)
    
    # Time-based Features
    daily_df['month'] = daily_df['date'].dt.month
    daily_df['day_of_week'] = daily_df['date'].dt.dayofweek
    
    # Rolling Statistics & Lags
    grouped = daily_df.groupby('city')
    
    daily_df['pm2_5_change_rate'] = grouped['pm2_5'].diff()
    daily_df['pm2_5_rolling_3d'] = grouped['pm2_5'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    daily_df['pm2_5_rolling_7d'] = grouped['pm2_5'].transform(lambda x: x.rolling(7, min_periods=1).mean())
    
    # Target Generation (Multi-step forecasting)
    daily_df['target_pm2_5_1d'] = grouped['pm2_5'].shift(-1)
    daily_df['target_pm2_5_2d'] = grouped['pm2_5'].shift(-2)
    daily_df['target_pm2_5_3d'] = grouped['pm2_5'].shift(-3)
    
    # Drop rows with NaN values
    final_df = daily_df.dropna().reset_index(drop=True)
    
    print("Feature engineering complete.")
    print(f"Final feature matrix shape: {final_df.shape}")
    return final_df

# Execute the pipeline to create features_df
features_df = build_feature_pipeline(master_dataset)

print("\nEngineered Features Preview:")
print(features_df[['city', 'date', 'pm2_5', 'pm2_5_rolling_3d', 'target_pm2_5_1d']].head())

In [7]:
import os
import hopsworks
from dotenv import load_dotenv

# Load credentials from the .env file
load_dotenv()

print("Authenticating with Hopsworks...")
# Explicitly targeting your newly created project
project = hopsworks.login(project="aqi_prediction_project") 
fs = project.get_feature_store()

print("Configuring Feature Group schema...")
aqi_fg = fs.get_or_create_feature_group(
    name="global_aqi_features",
    version=1,
    primary_key=["city", "date"],
    description="Daily aggregated AQI features and multi-day targets for global cities."
)

print("Uploading feature matrix to the Feature Store. This may take a few minutes...")
aqi_fg.insert(features_df)
print("Feature upload complete. Data is now securely stored in the cloud.")

Authenticating with Hopsworks...
2026-04-18 21:28:12,381 INFO: Initializing external client
2026-04-18 21:28:12,382 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443




To ensure compatibility please install the latest bug fix release matching the minor version of your backend (4.7) by running 'pip install hopsworks==4.7.*'


ProjectException: Could not find any project

In [ ]:
import hopsworks
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Fetch historical features and targets from the Feature Store
print("Connecting to Hopsworks Feature Store...")
project = hopsworks.login()
fs = project.get_feature_store()

print("Retrieving Feature Group...")
aqi_fg = fs.get_feature_group(name="global_aqi_features", version=1)
query = aqi_fg.select_all()

print("Creating Feature View and downloading training dataset...")
# The Feature View creates a logical grouping of features and defines the target variables
feature_view = fs.get_or_create_feature_view(
    name="global_aqi_view",
    version=1,
    description="Dataset for AQI forecasting",
    labels=["target_pm2_5_1d", "target_pm2_5_2d", "target_pm2_5_3d"],
    query=query
)

train_data, _ = feature_view.training_data(description="AQI baseline training data")

# 2. Data Preparation
# Sort by date to prevent temporal data leakage during the train/test split
train_data = train_data.sort_values(by='date')

# Isolate features (X) and the primary 1-day target (y)
X = train_data.drop(columns=['city', 'date', 'target_pm2_5_1d', 'target_pm2_5_2d', 'target_pm2_5_3d'])
y = train_data['target_pm2_5_1d']

# Sequential split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Train and Evaluate the Model
print("\nTraining Random Forest Regressor...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("Evaluating model performance...")
predictions = model.predict(X_test)
rmse = mean_squared_error(y_test, predictions, squared=False)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("\nModel Evaluation Metrics:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

# 4. Store the trained model in the Model Registry
print("\nExporting model artifact...")
model_dir = "aqi_model_dir"
os.makedirs(model_dir, exist_ok=True)
joblib.dump(model, f"{model_dir}/random_forest_aqi.pkl")

print("Registering model to Hopsworks Model Registry...")
mr = project.get_model_registry()
aqi_model = mr.python.create_model(
    name="aqi_forecasting_model",
    metrics={"RMSE": rmse, "MAE": mae, "R2": r2},
    description="Random Forest regressor predicting next-day PM2.5 levels."
)

aqi_model.save(model_dir)
print("Pipeline execution complete. Model successfully registered in the cloud.")